# 02 Build Silver Static Tables & Validation

**Project:** Metro Vancouver Transit Analytics  
**Source (Bronze):** `abfss://bronze@stransitanalytics.dfs.core.windows.net/gtfs_static`  
**Target (Silver):** `abfss://silver@stransitanalytics.dfs.core.windows.net`  

### Objectives (Step 2A):
1. Ingest raw GTFS text files from Bronze with standardized schemas and data types.
2. Build and write conformed analytical dimensions:
   - `dim_route` (Deduplicated routes)
   - `dim_stop` (Deduplicated stops with coordinates and accessibility)
   - `dim_trip` (Trips conformed and enriched with human-readable `direction`)
3. Build conformed scheduled event fact table:
   - `fact_scheduled_stop` (Stop times schedule enriched with `route_id` for star-schema joins)
4. Write conformed tables to Silver as **Parquet**.
5. Validate Silver tables for completeness, primary-key uniqueness, and foreign-key integrity.

In [ ]:
from pyspark.sql.functions import col, trim
import json

storage_account = "stransitanalytics"
bronze_base = f"abfss://bronze@{storage_account}.dfs.core.windows.net/gtfs_static"
silver_base = f"abfss://silver@{storage_account}.dfs.core.windows.net"

print("Bronze Base:", bronze_base)
print("Silver Base:", silver_base)

### 1. Build `dim_route`

In [ ]:
routes_raw = spark.read.option("header", True).option("inferSchema", True).csv(f"{bronze_base}/routes.txt")

dim_route = (
    routes_raw
    .select(
        trim(col("route_id").cast("string")).alias("route_id"),
        trim(col("route_short_name").cast("string")).alias("route_short_name"),
        trim(col("route_long_name").cast("string")).alias("route_long_name"),
        col("route_type").cast("int").alias("route_type")
    )
    .dropDuplicates(["route_id"])
)

dim_route.write.mode("overwrite").parquet(f"{silver_base}/dim_route")
print("dim_route written successfully.")
display(dim_route.limit(5))

### 2. Build `dim_stop`

In [ ]:
stops_raw = spark.read.option("header", True).option("inferSchema", True).csv(f"{bronze_base}/stops.txt")

dim_stop = (
    stops_raw
    .select(
        trim(col("stop_id").cast("string")).alias("stop_id"),
        trim(col("stop_code").cast("string")).alias("stop_code"),
        trim(col("stop_name").cast("string")).alias("stop_name"),
        col("stop_lat").cast("double").alias("stop_lat"),
        col("stop_lon").cast("double").alias("stop_lon"),
        trim(col("zone_id").cast("string")).alias("zone_id"),
        col("wheelchair_boarding").cast("int").alias("wheelchair_boarding")
    )
    .dropDuplicates(["stop_id"])
)

dim_stop.write.mode("overwrite").parquet(f"{silver_base}/dim_stop")
print("dim_stop written successfully.")
display(dim_stop.limit(5))

### 3. Build `dim_trip` (Enriched with human-readable `direction`)

In [ ]:
trips_raw = spark.read.option("header", True).option("inferSchema", True).csv(f"{bronze_base}/trips.txt")
directions_raw = spark.read.option("header", True).option("inferSchema", True).csv(f"{bronze_base}/directions.txt")

directions_lookup = (
    directions_raw
    .select(
        trim(col("route_id").cast("string")).alias("route_id"),
        trim(col("direction_id").cast("string")).alias("direction_id"),
        trim(col("direction").cast("string")).alias("direction")
    )
    .dropDuplicates(["route_id", "direction_id"])
)

trips_clean = (
    trips_raw
    .select(
        trim(col("trip_id").cast("string")).alias("trip_id"),
        trim(col("route_id").cast("string")).alias("route_id"),
        trim(col("service_id").cast("string")).alias("service_id"),
        trim(col("direction_id").cast("string")).alias("direction_id"),
        trim(col("trip_headsign").cast("string")).alias("trip_headsign"),
        trim(col("shape_id").cast("string")).alias("shape_id")
    )
    .dropDuplicates(["trip_id"])
)

dim_trip = (
    trips_clean
    .join(directions_lookup, on=["route_id", "direction_id"], how="left")
    .select(
        "trip_id",
        "route_id",
        "service_id",
        "direction_id",
        "direction",
        "trip_headsign",
        "shape_id"
    )
)

dim_trip.write.mode("overwrite").parquet(f"{silver_base}/dim_trip")
print("dim_trip written successfully.")
display(dim_trip.limit(5))

### 4. Build `fact_scheduled_stop` (Enriched with `route_id`)

In [ ]:
stop_times_raw = spark.read.option("header", True).option("inferSchema", True).csv(f"{bronze_base}/stop_times.txt")
trip_route_lookup = trips_clean.select("trip_id", "route_id")

fact_scheduled_stop = (
    stop_times_raw
    .select(
        trim(col("trip_id").cast("string")).alias("trip_id"),
        trim(col("stop_id").cast("string")).alias("stop_id"),
        col("stop_sequence").cast("int").alias("stop_sequence"),
        trim(col("arrival_time").cast("string")).alias("arrival_time"),
        trim(col("departure_time").cast("string")).alias("departure_time"),
        col("pickup_type").cast("int").alias("pickup_type"),
        col("drop_off_type").cast("int").alias("drop_off_type")
    )
    .join(trip_route_lookup, on="trip_id", how="left")
    .select(
        "trip_id",
        "route_id",
        "stop_id",
        "stop_sequence",
        "arrival_time",
        "departure_time",
        "pickup_type",
        "drop_off_type"
    )
)

fact_scheduled_stop.write.mode("overwrite").parquet(f"{silver_base}/fact_scheduled_stop")
print("fact_scheduled_stop written successfully.")
display(fact_scheduled_stop.limit(5))

### 5. Silver Validation & Quality Reporting
Read back the 4 Parquet tables and validate:
1. Row counts
2. Null counts on primary & foreign keys
3. Primary key duplicate checks
4. Referential integrity & orphan checks

In [ ]:
silver_routes = spark.read.parquet(f"{silver_base}/dim_route")
silver_stops = spark.read.parquet(f"{silver_base}/dim_stop")
silver_trips = spark.read.parquet(f"{silver_base}/dim_trip")
silver_facts = spark.read.parquet(f"{silver_base}/fact_scheduled_stop")

# Row Counts
cnt_routes = silver_routes.count()
cnt_stops = silver_stops.count()
cnt_trips = silver_trips.count()
cnt_facts = silver_facts.count()

print("=== Silver Table Row Counts ===")
print(f"dim_route:           {cnt_routes:,}")
print(f"dim_stop:            {cnt_stops:,}")
print(f"dim_trip:            {cnt_trips:,}")
print(f"fact_scheduled_stop: {cnt_facts:,}")

# Null Checks
print("\n=== Primary/Foreign Key Null Counts ===")
print("dim_route null route_id:            ", silver_routes.filter(col("route_id").isNull()).count())
print("dim_stop null stop_id:              ", silver_stops.filter(col("stop_id").isNull()).count())
print("dim_trip null trip_id:              ", silver_trips.filter(col("trip_id").isNull()).count())
print("dim_trip null route_id:             ", silver_trips.filter(col("route_id").isNull()).count())
print("fact_scheduled_stop null trip_id:   ", silver_facts.filter(col("trip_id").isNull()).count())
print("fact_scheduled_stop null route_id:  ", silver_facts.filter(col("route_id").isNull()).count())
print("fact_scheduled_stop null stop_id:   ", silver_facts.filter(col("stop_id").isNull()).count())
print("fact_scheduled_stop null stop_seq:  ", silver_facts.filter(col("stop_sequence").isNull()).count())

# Duplicate PK Checks
print("\n=== Primary Key Duplicate Checks ===")
print("dim_route duplicate route_id:       ", cnt_routes - silver_routes.select("route_id").distinct().count())
print("dim_stop duplicate stop_id:         ", cnt_stops - silver_stops.select("stop_id").distinct().count())
print("dim_trip duplicate trip_id:         ", cnt_trips - silver_trips.select("trip_id").distinct().count())

# Foreign Key / Orphan Checks
print("\n=== Referential Integrity / Orphan Checks ===")
orphan_trip_route = silver_trips.join(silver_routes, on="route_id", how="left_anti").count()
orphan_fact_trip = silver_facts.join(silver_trips, on="trip_id", how="left_anti").count()
orphan_fact_route = silver_facts.join(silver_routes, on="route_id", how="left_anti").count()
orphan_fact_stop = silver_facts.join(silver_stops, on="stop_id", how="left_anti").count()

print(f"dim_trip.route_id -> dim_route.route_id orphans:             {orphan_trip_route}")
print(f"fact_scheduled_stop.trip_id -> dim_trip.trip_id orphans:     {orphan_fact_trip}")
print(f"fact_scheduled_stop.route_id -> dim_route.route_id orphans:   {orphan_fact_route}")
print(f"fact_scheduled_stop.stop_id -> dim_stop.stop_id orphans:     {orphan_fact_stop}")